In [ ]:
import sqlite3
import json
import pandas as pd
from pathlib import Path

DB_PATH = Path("./data/web-questionnaires.db")

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def list_sessions():
    with get_connection() as conn:
        rows = conn.execute(
            "SELECT id, code, name, created_at FROM sessions ORDER BY id"
        ).fetchall()
        return pd.DataFrame(rows, columns=["id", "code", "name", "created_at"])


def list_submissions(session_code=None, session_id=None):
    with get_connection() as conn:
        if session_code:
            session = conn.execute(
                "SELECT id FROM sessions WHERE code = ?", (session_code,)
            ).fetchone()
            if not session:
                return f"No session with code {session_code}"
            session_id = session["id"]
        elif session_id is None:
            return "Provide either session_code or session_id"

        rows = conn.execute(
            "SELECT questionnaire, submitted_at, responses FROM submissions WHERE session_id = ?",
            (session_id,)
        ).fetchall()
        if not rows:
            return "No submissions yet."
        df = pd.DataFrame(rows, columns=["questionnaire", "submitted_at", "responses"])
        df["responses"] = df["responses"].apply(json.loads)  # expand JSON
        return df


def get_questionnaire(session_code, questionnaire_name):
    with get_connection() as conn:
        session = conn.execute(
            "SELECT id FROM sessions WHERE code = ?", (session_code,)
        ).fetchone()
        if not session:
            raise ValueError(f"Session {session_code} not found")
        row = conn.execute(
            "SELECT responses FROM submissions WHERE session_id = ? AND questionnaire = ?",
            (session["id"], questionnaire_name)
        ).fetchone()
        if not row:
            return f"No submission for {questionnaire_name}"
        events = json.loads(row["responses"])
        return pd.DataFrame(events)

In [2]:
list_sessions()

,id,code,name,created_at
0,1,test,,2026-06-13 10:55:56
1,2,hello,,2026-06-13 11:17:30


In [4]:
list_submissions(session_code="hello")

,questionnaire,submitted_at,responses
0,sam,2026-06-13 11:17:35,"[{'t': 1781349453250, 'line': None, 'imagePos'..."


In [10]:
sam = get_questionnaire("hello", "sam")
sam

,t,line,imagePos
0,1781349453250,NaN,NaN
1,1781349454448,0.0,4.0
2,1781349454955,1.0,4.0
